# 05 -- Preprocessing and stratified splits

Build a train-only sklearn pipeline and create a stratified 80/20 holdout plus
stratified 5-fold CV indices.

Pipeline steps (fit on training fold / train split only):
1. Missing indicators for high-signal sparse fields
2. Median imputation
3. Winsorise / cap outliers at training quantiles (1st / 99th)

All features in `train_features.parquet` are numeric (categoricals were
aggregated in Stage 3), so no categorical encoder is needed here.

Split indices are saved for reproducible modelling.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    HOLDOUT_SIZE,
    ID_COL,
    N_CV_FOLDS,
    PROCESSED_DATA_DIR,
    RANDOM_SEED,
    SPLIT_INDICES_PATH,
    TARGET_COL,
    TRAIN_FEATURES_PATH,
)

print("seed", RANDOM_SEED, "holdout", HOLDOUT_SIZE, "cv folds", N_CV_FOLDS)
print("features", TRAIN_FEATURES_PATH)

In [ ]:
df = pd.read_parquet(TRAIN_FEATURES_PATH)
feature_cols = [c for c in df.columns if c not in {ID_COL, TARGET_COL}]
X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int)
ids = df[ID_COL]

print("shape", X.shape)
print("target rate", round(y.mean(), 4))
print("n features", len(feature_cols))

miss = (X.isna().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 10))
sns.barplot(x=miss.values, y=miss.index, ax=ax, color="salmon")
ax.set_xlabel("% missing")
ax.set_title("Feature missingness before preprocess")
plt.tight_layout()
plt.show()

## Stratified holdout + CV indices

Holdout is carved first. CV folds are defined **only on the train partition**.

In [ ]:
idx = np.arange(len(df))
train_idx, holdout_idx = train_test_split(
    idx,
    test_size=HOLDOUT_SIZE,
    stratify=y,
    random_state=RANDOM_SEED,
)

print("train", len(train_idx), "holdout", len(holdout_idx))
print("train target rate", round(y.iloc[train_idx].mean(), 4))
print("holdout target rate", round(y.iloc[holdout_idx].mean(), 4))

skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
cv_folds = []
for fold_i, (tr, va) in enumerate(skf.split(train_idx, y.iloc[train_idx])):
    # Map fold-local positions back to global row indices
    global_tr = train_idx[tr].tolist()
    global_va = train_idx[va].tolist()
    cv_folds.append({"fold": fold_i, "train_idx": global_tr, "valid_idx": global_va})
    print(
        f"fold {fold_i}: train={len(global_tr)} valid={len(global_va)} "
        f"valid_rate={y.iloc[global_va].mean():.4f}"
    )

# No overlap between holdout and train/CV
assert set(train_idx).isdisjoint(set(holdout_idx))
assert len(set(train_idx) | set(holdout_idx)) == len(df)
print("split integrity ok")

## Preprocess transformers (train-fit only)

In [ ]:
# High-signal / high-missing fields get explicit missing indicators
HIGH_SIGNAL_MISSING = [
    "app_ext_source_1",
    "app_ext_source_2",
    "app_ext_source_3",
    "app_years_employed",
    "cc_utilisation_mean",
    "cc_utilisation_max",
    "inst_late_rate",
    "inst_late_rate_3m",
    "inst_late_rate_6m",
    "inst_late_rate_12m",
    "bur_utilisation",
    "bur_bb_dpd_share_mean",
    "prev_refused_rate",
    "prev_approved_rate",
]
HIGH_SIGNAL_MISSING = [c for c in HIGH_SIGNAL_MISSING if c in feature_cols]
print("missing-indicator cols:", HIGH_SIGNAL_MISSING)


class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    """Add binary missing flags for selected columns; keep original NaNs."""

    def __init__(self, columns: list[str]):
        self.columns = columns

    def fit(self, X, y=None):
        self.columns_ = [c for c in self.columns if c in X.columns]
        return self

    def transform(self, X):
        out = X.copy()
        for c in self.columns_:
            out[f"{c}_was_missing"] = out[c].isna().astype("int8")
        return out


class QuantileCapper(BaseEstimator, TransformerMixin):
    """Winsorise numeric columns using training quantiles."""

    def __init__(self, lower: float = 0.01, upper: float = 0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        num = X.select_dtypes(include=[np.number])
        # Skip binary indicator columns
        cols = [c for c in num.columns if not c.endswith("_was_missing")]
        self.bounds_ = {}
        for c in cols:
            lo, hi = num[c].quantile([self.lower, self.upper])
            self.bounds_[c] = (float(lo), float(hi))
        return self

    def transform(self, X):
        out = X.copy()
        for c, (lo, hi) in self.bounds_.items():
            if c in out.columns:
                out[c] = out[c].clip(lo, hi)
        return out


class MedianImputerFrame(BaseEstimator, TransformerMixin):
    """Median-impute all numeric columns; return a DataFrame with same columns."""

    def fit(self, X, y=None):
        self.columns_ = list(X.columns)
        self.imputer_ = SimpleImputer(strategy="median")
        self.imputer_.fit(X[self.columns_])
        return self

    def transform(self, X):
        arr = self.imputer_.transform(X[self.columns_])
        return pd.DataFrame(arr, columns=self.columns_, index=X.index)


preprocess = Pipeline(
    steps=[
        ("missing_flags", MissingIndicatorAdder(HIGH_SIGNAL_MISSING)),
        ("impute", MedianImputerFrame()),
        ("cap", QuantileCapper(lower=0.01, upper=0.99)),
    ]
)

## Fit on train only -- before / after checks

In [ ]:
X_train = X.iloc[train_idx].copy()
X_hold = X.iloc[holdout_idx].copy()

preprocess.fit(X_train)
X_train_t = preprocess.transform(X_train)
X_hold_t = preprocess.transform(X_hold)

print("train transformed", X_train_t.shape)
print("holdout transformed", X_hold_t.shape)
print("train remaining NaNs", int(X_train_t.isna().sum().sum()))
print("holdout remaining NaNs", int(X_hold_t.isna().sum().sum()))
assert X_train_t.isna().sum().sum() == 0
assert X_hold_t.isna().sum().sum() == 0

# Show capping effect on a heavy-tailed feature
col = "app_amt_income_total"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(X_train[col].dropna().clip(upper=X_train[col].quantile(0.995)), bins=40, ax=axes[0], color="steelblue")
axes[0].set_title(f"{col} before (plot clipped for visibility)")
sns.histplot(X_train_t[col], bins=40, ax=axes[1], color="teal")
axes[1].set_title(f"{col} after median impute + p01/p99 cap")
plt.tight_layout()
plt.show()

lo, hi = preprocess.named_steps["cap"].bounds_[col]
print(f"train quantile bounds for {col}: [{lo:.1f}, {hi:.1f}]")

In [ ]:
# Missing-indicator rates on train
flag_cols = [c for c in X_train_t.columns if c.endswith("_was_missing")]
rates = X_train_t[flag_cols].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=rates.values, y=rates.index, ax=ax, color="salmon")
ax.set_xlabel("rate")
ax.set_title("Missing-indicator rates (train)")
plt.tight_layout()
plt.show()
display(rates.to_frame("rate"))

In [ ]:
# Fold-wise leakage check: fit preprocess on fold train, transform valid -- no NaNs
for fold in cv_folds[:2]:  # spot-check first two folds
    pipe = Pipeline(
        steps=[
            ("missing_flags", MissingIndicatorAdder(HIGH_SIGNAL_MISSING)),
            ("impute", MedianImputerFrame()),
            ("cap", QuantileCapper(lower=0.01, upper=0.99)),
        ]
    )
    Xt = X.iloc[fold["train_idx"]]
    Xv = X.iloc[fold["valid_idx"]]
    pipe.fit(Xt)
    Xt_t = pipe.transform(Xt)
    Xv_t = pipe.transform(Xv)
    assert Xt_t.isna().sum().sum() == 0
    assert Xv_t.isna().sum().sum() == 0
    # Cap bounds must come from fold train only (holdout/other folds not involved)
    print(f"fold {fold['fold']}: transform ok, n_features={Xt_t.shape[1]}")
print("CV fold preprocess spot-check passed")

## Save split indices

In [ ]:
payload = {
    "random_seed": RANDOM_SEED,
    "holdout_size": HOLDOUT_SIZE,
    "n_cv_folds": N_CV_FOLDS,
    "n_rows": int(len(df)),
    "train_idx": train_idx.tolist(),
    "holdout_idx": holdout_idx.tolist(),
    "cv_folds": cv_folds,
    "feature_cols": feature_cols,
    "high_signal_missing": HIGH_SIGNAL_MISSING,
    "train_target_rate": float(y.iloc[train_idx].mean()),
    "holdout_target_rate": float(y.iloc[holdout_idx].mean()),
}

SPLIT_INDICES_PATH.parent.mkdir(parents=True, exist_ok=True)
SPLIT_INDICES_PATH.write_text(json.dumps(payload), encoding="ascii")
print("wrote", SPLIT_INDICES_PATH)
print("train", len(payload["train_idx"]), "holdout", len(payload["holdout_idx"]))
print("cv folds", len(payload["cv_folds"]))

## Summary

- Stratified 80/20 holdout; target rates matched across splits.
- Stratified 5-fold CV indices on the train partition only.
- Preprocess: missing indicators (high-signal fields) -> median impute -> p01/p99 cap.
- All fitting on train / fold-train only; holdout and fold-valid are transform-only.
- Split indices: `data/processed/split_indices.json`.

Next: port helpers into `src/preprocess.py` so modelling can `fit_transform` / `transform` reproducibly.